<center>
<img src="../../img/ods_stickers.jpg" />
    
## [mlcourse.ai](https://mlcourse.ai) – دورة التعلم الآلي المفتوحة 
### <center> المؤلف: إيرينا كنيازيفا، لقب ODS Slack: iknyazeva
    
## <center> البرنامج التعليمي
### <center> "التعامل مع مجموعات البيانات المختلفة باستخدام DASK وتجربة القليل من DASK ML"



## لماذا أحتاج إلى DASK؟
يوفر Dask مجموعات Array وBag وDataFrame عالية المستوى تحاكي NumPy والقوائم وPandas ولكن يمكنها العمل بالتوازي على مجموعات البيانات التي لا تتلاءم مع الذاكرة الرئيسية. تعد مجموعات Dask عالية المستوى بمثابة بدائل لـ NumPy وPandas لمجموعات البيانات الكبيرة.
## أنت بالتأكيد بحاجة إلى DASK إذا
إذا كان حجم المشكلة قريبًا من حدود ذاكرة الوصول العشوائي (RAM)، ولكنه يناسب القرص
## قائمة القراءة
يعتمد هذا الدفتر بشكل أساسي على هذه المصادر الثلاثة
- [برنامج تعليمي آخر من تحليلات vidhya](https://www.analyticsvidhya.com/blog/2018/08/dask-big-datasets-machine_learning-python/)
- [مأخوذة من نحو علم البيانات](https://towardsdatascience.com/trying-out-dask-dataframes-in-python-for-fast-data-analysis-in-parallel-aa960c18a915)
- [دورة DataCamp](https://campus.datacamp.com/courses/parallel-computing-with-dask/)
- [توثيق Dask](https://docs.dask.org/en/latest/) 


In [ ]:
import gc
import os
import time
import warnings

import numpy as np
import pandas as pd
import psutil
from dask import delayed

warnings.filterwarnings("ignore")


لنكتب وظيفة صغيرة لتتبع الذاكرة التي تتطلب عملية بايثون


In [ ]:
def memory_footprint():
    mem = psutil.Process(os.getpid()).memory_info().rss
    return mem / 1024 ** 2

In [ ]:
before = memory_footprint()
print(f"Memory used before is {round(before,2)} MB")

In [ ]:
N = (1024 ** 2) // 8
x = np.random.randn(50 * N)
after = memory_footprint()
print(f"Memory used after is {round(after,2)} MB")


يحسب، ولكن لا يربط النتيجة بمتغير يخصص ذاكرة إضافية


In [ ]:
x ** 2
after1 = memory_footprint()
print(f" Extra memory obtained after computation {round(after1,2)} MB")


## صفائف Dask
يطبق Dask Array مجموعة فرعية من واجهة NumPy ndarray باستخدام خوارزميات محظورة، مما يؤدي إلى تقطيع المصفوفة الكبيرة إلى العديد من المصفوفات الصغيرة. يتيح لنا ذلك الحساب على صفائف أكبر من الذاكرة باستخدام جميع مراكزنا. نقوم بتنسيق هذه الخوارزميات المحظورة باستخدام الرسوم البيانية Dask.[وثائق مصفوفة dask](http://docs.dask.org/en/latest/array.html)
<center>
<img src="@@KEEP_00008@@ />
يوجد في dask ثلاثة هياكل رئيسية: مصفوفة dask (استنادًا إلى مصفوفة numpy)، وإطار بيانات dask (استنادًا إلى إطار بيانات الباندا) وأكياس dask (للبيانات غير المنظمة كنص).


In [ ]:
import dask.array as da

y = da.from_array(x, chunks=len(x) // 4)
print("Dask arrays require little memory:", memory_footprint() - after1)

In [ ]:
import time

t_start = time.time()
x.mean()
t_end = time.time()
print("Compute mean value of this numpy array \n")
print(
    "Elapsed time for compute mean of numpy array (ms):",
    round((t_end - t_start) * 1000),
)

In [ ]:
t_start = time.time()
y.mean().compute()
t_end = time.time()
print("Compute the same with dask \n")
print(
    "Elapsed time for compute mean of dask array (ms):", round((t_end - t_start) * 1000)
)

في الواقع، لن يتم استخدام هذا المثال عمليًا أبدًا، لأنه إذا كان الرقم الخاص بك موجودًا بالفعل في الذاكرة، فإن أي تقسيم سيؤدي دائمًا إلى زيادة الوقت الحسابي. ولكن إذا كنت بحاجة إلى معالجة البيانات من HDF5 أو NetCDF أو مجموعة كبيرة من الملفات غير المرغوب فيها من القرص، فقد يكون ذلك مفيدًا للغاية



## العمليات المؤجلة مع dask
لكن dask قد يكون مفيدًا للبيانات الصغيرة ذات الحسابات المتأخرة. يمكن بسهولة موازاة الحساب. دعونا نرى المثال مع مجموعة numpy السابقة لدينا   


In [ ]:
def f(z):
    return np.sqrt(z + 4)


def g(y):
    return y - 3


def h(x):
    return x ** 2


time_start = time.time()
x = np.random.randn(50 * N)
y = h(x)
z = g(x)
w = f(z + y)
time_end = time.time()
print(
    "Elapsed time for compute complex functions with numpy array (ms):",
    round((time_end - time_start) * 1000),
)

In [ ]:
y = delayed(h)(x)
z = delayed(g)(x)
w = delayed(f)(z + y)
print("After we get dask delayed object", w)
time_start = time.time()
w.compute()
time_end = time.time()
print(
    "Elapsed time for compute complex functions with numpy array with dask delayed (ms):",
    round((time_end - time_start) * 1000),
)


من السهل أن نفهم سبب انخفاض وقت الحساب باستخدام الرسم البياني الحسابي. دعونا نفعل ذلك بالطريقة الثانية لإدخال وظائف التأخير


In [ ]:
@delayed
def f(z):
    return np.sqrt(z + 4)


@delayed
def g(y):
    return y - 3


@delayed
def h(x):
    return x ** 2


y = h(x)
z = g(x)
w = f(z + y)
w.visualize()


## إطار بيانات داسك 
تقوم Dask DataFrames بتنسيق العديد من إطارات/سلاسل بيانات Pandas المرتبة على طول الفهرس. يتم تقسيم Dask DataFrame حسب الصفوف، حيث يتم تجميع الصفوف حسب قيمة الفهرس لتحقيق الكفاءة. قد توجد كائنات Pandas هذه على القرص أو على أجهزة أخرى.
(راجع الوثائق)[http://docs.dask.org/en/latest/dataframe.html]
<center>
<img src="@@KEEP_00010@@ width="40%" height="40% />



نستخدم هنا الملف `athlete_events.csv` من [مجموعة بيانات Kaggle هذه](https://www.kaggle.com/heesoo37/120-years-of-olympic-history-athletes-and-results)


In [ ]:
import dask.dataframe as dd

In [ ]:
print("Let's return to start of our ML journey\n")
print("Load olympic dataset \n")
PATH = "../../data/athlete_events.csv"

In [ ]:
df = pd.read_csv(PATH)
df.head()

In [ ]:
m1 = memory_footprint()
dask_df = dd.read_csv(PATH)
m2 = memory_footprint()
print("Dask do not allocate memory after creation:", m2 - m1)

In [ ]:
print("But we could see data as in pandas dataframe:")
dask_df.head()

In [ ]:
# building delayed  computation
print(
    "We can do many operation the same way as in pandas, but without loading all data in memory \n "
)
sex_distr = (
    dask_df.loc[dask_df["Games"].str.contains("1996")].groupby("Sex")["Age"].min()
)

In [ ]:
print(
    "Here we done selecting and aggregation exactly the same way as we did in pandas \n"
)
print("But there is not any computation, we create dask structure \ n")
sex_distr

In [ ]:
%%time
print(
    "Computation is time consuming, but we remember that we dont't need to load all data in memory for this computation \n"
)
print(sex_distr.compute())

In [ ]:
%%time
print("Pandas of course more effective \n")
print(df.loc[df["Games"].str.contains("1996")].groupby("Sex")["Age"].min())


### التوافق مع واجهة برمجة تطبيقات Pandas
- غير متوفر في dask.dataframe:
    * بعض تنسيقات الملفات غير المدعومة (على سبيل المثال، .xls، .zip،...)
    * الفرز
- متوفر في dask.dataframe:
    * الفهرسة والاختيار وإعادة الفهرسة
    * التجميعات: .sum()، .mean()، .std()، .min()، .max() إلخ.
    * التجميع باستخدام .groupby()
    * تحويل التاريخ والوقت باستخدام dd.to_datetime()



### قراءة مجموعات من الملفات إلى dask dataframe
على سبيل المثال لقد اتخذت مشروع أليكا.  أرشيف Capstone_user_identification [الرابط](https://drive.google.com/open?id=1AU3M_mFPofbfhFQa_Bktozq_vFREkWJA) (~7 ميجا بايت، البيانات غير المضغوطة ~60 ميجا بايت).


In [ ]:
PATH_TO_DATA = "../../data/capstone_user_identification"

In [ ]:
print("We can load all files in single dataframe \n")
print("Your dont't need this in Alica project, just an example \n ")
user10dask = dd.read_csv(os.path.join(PATH_TO_DATA, "10users/*.csv"))

In [ ]:
print("We can look at the data")
print(user10dask)
user10dask.tail()

In [ ]:
print(
    "Let's see what happens if we want to count all sites (it could seen as a one more way for dictionary creation) \n"
)
count_sites = user10dask.groupby("site")["site"].count()

In [ ]:
print("If we visualize this structure we'll see the picture of computation \n")
count_sites.visualize()

In [ ]:
%%time
count_sites.compute().sort_values(ascending=False)[:20]


## ملفات JSON في أكياس Daskتنفذ Dask Bag عمليات مثل الخريطة والتصفية والطي والتجميع على مجموعات من كائنات Python. يقوم بذلك بالتوازي مع مساحة ذاكرة صغيرة باستخدام مكررات Python. وهو مشابه لإصدار موازٍ من PyToolz أو إصدار Pythonic من PySpark RDD.[وثائق حقيبة Dask](http://docs.dask.org/en/latest/bag-overview.html)
غالبًا ما تُستخدم أكياس Dask لموازنة العمليات الحسابية البسيطة على البيانات غير المنظمة أو شبه المنظمة مثل البيانات النصية أو ملفات السجل أو سجلات JSON أو كائنات Python المحددة من قبل المستخدم.
 
دعونا نرى مثالاً على بياناتنا المتوسطة من [هذه المنافسة](https://www.kaggle.com/c/how-good-is-your-medium-article/data).


In [ ]:
import json

import dask.bag as db

In [ ]:
print("Path to our medium data \n")
PATH = "../../data/kaggle_medium"
print(PATH)

In [ ]:
print("Wrap train json to dask bag format \n")
items = db.read_text(os.path.join(PATH, "train.json"))
items

In [ ]:
%%time
print("Let's look at one example \n")
print(items.take(1))

In [ ]:
print("We can parse date with json library and get dict like object \n")
dict_items = items.map(json.loads)
print(type(dict_items))

In [ ]:
dict_items.take(1)

In [ ]:
print("We can take any key from all records \n")
title_bag = dict_items.pluck("title")
print("With take method we received tuple of objects \n")
print(title_bag.take(3))


يمكننا كتابة أي وظيفة لمعالجة البيانات وتطبيقها باستخدام وظيفة الخريطة


In [ ]:
def clean_title(text):

    import string

    cut_set = set(string.punctuation)
    cut_set.update(["”", "—", "…", "“", "⌘", "❤", "+", "®", "➜", "¬", "–"])
    text = text.translate(text.maketrans("".join(cut_set), " " * len(cut_set)))
    text = text.lower()
    return text

In [ ]:
title_bag = dict_items.pluck("title").map(clean_title)

In [ ]:
title_bag.take(3)


معالجة العلامات الوصفية


In [ ]:
meta_tags_bag = dict_items.pluck("meta_tags")
test_meta = meta_tags_bag.take(3)

In [ ]:
test_meta[1]

In [ ]:
def clean_meta_tags(meta):
    author = meta["author"].strip()
    min_reads = int(meta["twitter:data1"].split()[0])
    return {"author": author, "min_reads": min_reads}

In [ ]:
meta_tags_bag = meta_tags_bag.map(clean_meta_tags)

In [ ]:
meta_tags_bag.take(1)


### الجمع بين كل شيء معا


In [ ]:
%%time
# content_bag = dict_items.pluck('content').map(clean_content)
title_bag = dict_items.pluck("title").map(clean_title)
published_bag = dict_items.pluck("published").map(lambda x: x["$date"])
meta_bag = dict_items.pluck("meta_tags").map(clean_meta_tags)
domain_bag = dict_items.pluck("domain")

In [ ]:
@delayed
def combine_to_df(list_dict):

    list_df = [pd.DataFrame(dict_) for dict_ in list_dict]
    return pd.concat(list_df, axis=1)

In [ ]:
combined = combine_to_df([published_bag, meta_bag, domain_bag])
combined.visualize()

In [ ]:
# It takes time, around a minute
from dask.diagnostics import ProgressBar

with ProgressBar():
    df = combined.compute()
df.columns = ["published", "Author", "min_reads", "domain"]
df.head()

In [ ]:
print("We can create dask dataframe from pandas \n")
dd_no_content = dd.from_pandas(df, npartitions=4)

In [ ]:
dd_no_content

In [ ]:
%%time
print(
    "Transform published column to datetime as we did with pandas, it will by slightly slowly than in pandas \n"
)
df["published"] = pd.to_datetime(df.published, format="%Y-%m-%dT%H:%M:%S.%fZ")

In [ ]:
%%time
print("Transform published column to datetime  with pandas, \n")
dd_no_content["published"] = dd.to_datetime(
    dd_no_content.published, format="%Y-%m-%dT%H:%M:%S.%fZ"
).compute()

In [ ]:
dd_no_content.head()

In [ ]:
print(
    "We can apply function with mixed transformation to dask dataframe written for pandas df without changes \n"
)


def additional_time_features_df(
    df, to_cat_cols=["Author", "domain", "month", "year", "day_of_week"]
):

    df["month"] = df["published"].apply(lambda ts: ts.month)
    df["year"] = df["published"].apply(lambda ts: ts.year)
    hour = df["published"].apply(lambda ts: ts.hour)
    df["hour"] = hour
    df["morning"] = ((hour >= 7) & (hour <= 11)).astype("float64")
    df["day"] = ((hour >= 12) & (hour <= 18)).astype("int")
    df["evening"] = ((hour >= 19) & (hour <= 23)).astype("int")
    df["night"] = ((hour >= 0) & (hour <= 6)).astype("int")
    df["sin_hour"] = np.sin(2 * np.pi * df["hour"] / 24)
    df["cos_hour"] = np.cos(2 * np.pi * df["hour"] / 24)
    df = df.drop(["hour"], axis=1)
    day_of_week = df["published"].dt.dayofweek.astype("int")
    df["day_of_week"] = day_of_week
    df["weekend"] = (day_of_week >= 5).astype("int")
    # turn to categorical
    df[to_cat_cols] = df[to_cat_cols].astype("category")

    return df

In [ ]:
%%time
df_medium_train = additional_time_features_df(df.copy())

In [ ]:
dd_medium_train = additional_time_features_df(dd_no_content)

In [ ]:
%%time
dd_medium_train.compute()


## داسك مل
يوفر Dask ML خوارزميات تعلم آلي قابلة للتطوير في لغة بايثون ومتوافقة مع scikit-learn. دعونا نفهم أولاً كيف يتعامل scikit-learn مع العمليات الحسابية، ثم سننظر في كيفية قيام Dask بتنفيذ هذه العمليات بشكل مختلف. راجع دروس dask-ml: [أمثلة من dask ml](http://ml.dask.org/examples.html)
تحتاج إلى تثبيت dask-ml في البداية 
هناك جزأين رئيسيين في dask ml:
    - أساليب التعامل مع مجموعات البيانات الكبيرة 
    - أساليب التعامل مع النماذج الكبيرة



### التعامل مع النموذج الكبير مع توزيع dask
كان أكبر نموذج من الدورة التدريبية لدينا عبارة عن مجموعة عشوائية من البيانات النصية في الأسبوع مع مهمة Random Forest. أدناه أقوم بإعادة إنتاج جزء من مهمتنا، لكنني قمت بتقليل الأعداد والحد الأقصى من الميزات في Count Vectorizer، ولكن يمكنك التحقق من المعلمات الأصلية.
هنا نستخدم الملف [`movie_reviews_train.csv`](https://drive.google.com/file/d/1WDz3EB0MMuQUuUTwZ30c4JJrN8d9shAW/view?usp=sharing).


In [ ]:
# Download data
df = pd.read_csv("../../data/movie_reviews_train.csv", nrows=5000)

# Split data to train and test
X_text = df["text"]
y_text = df["label"]

# Classes counts
df.label.value_counts()

In [ ]:
from sklearn.feature_extraction.text import CountVectorizer
from sklearn.linear_model import LogisticRegression
from sklearn.model_selection import GridSearchCV, StratifiedKFold
from sklearn.pipeline import Pipeline

# Split on 3 folds
skf = StratifiedKFold(n_splits=3, shuffle=True, random_state=17)

# In Pipeline we will modify the text and train logistic regression
classifier = Pipeline(
    [
        ("vectorizer", CountVectorizer(max_features=500, ngram_range=(1, 3))),
        ("clf", LogisticRegression(random_state=17)),
    ]
)

In [ ]:
%%time
parameters = {"clf__C": (0.1, 1, 10, 100)}
grid_search = GridSearchCV(classifier, parameters, scoring="roc_auc", cv=skf)
grid_search = grid_search.fit(X_text, y_text)

In [ ]:
grid_search.best_score_


### استبدل joblib بـ dask
في هذا النهج، كل ما يتعين علينا القيام به هو استبدال joblib إلى dask الموزعة. نحتاج إلى تهيئة العميل الموزع وتغيير الواجهة الخلفية


In [ ]:
from dask.distributed import Client
%%time
from sklearn.externals import joblib

client = Client()
parameters = {"clf__C": (0.1, 1, 10, 100)}
grid_search = GridSearchCV(classifier, parameters, scoring="roc_auc", cv=skf)

t_start = time.time()

with joblib.parallel_backend("dask"):
    grid_search.fit(X_text, y_text)
t_end = time.time()
print("Elapsed time for grid_search with joblib replace (s):", round((t_end - t_start)))

In [ ]:
grid_search.best_score_

### استبدال بحث الشبكة بـ dask
بالتوازي مع Gridsearch CV في sklearn، يوفر Dask مكتبة تسمى Dask-search CV (تم تضمين السيرة الذاتية لـ Dask-search الآن في Dask ML). فهو يدمج الخطوات بحيث يكون هناك تكرار أقل. فيما يلي خطوات التثبيت لـ Dask-search. نحن بحاجة إلى تثبيته بشكل منفصل


In [ ]:
# pip3 install dask-searchcv
import dask_searchcv as dcv


يمكننا استخدام خطوط الأنابيب في بحث شبكة dask، ووفقًا للوثائق، يجب أن نستخدم dask مع خطوط الأنابيب التي تحتوي على العديد من العمليات التي يمكن موازنتها، وخاصة اتحاد الميزات المضمن، لكنني حاولت وحصلت على خطأ نتيجة لذلك... على أي حال، لا يمكن موازاة العمليات التي تستغرق وقتًا طويلاً مثل CountVectorizer، لذلك هنا بحث الشبكة من dask فقط للمصنف [الوثائق](https://dask-searchcv.readthedocs.io/en/latest/). 


In [ ]:
%%time
vect = CountVectorizer(max_features=500, ngram_range=(1, 3))
Xvect = vect.fit_transform(X_text)

In [ ]:
lr = LogisticRegression()
parameters = {"C": (0.1, 1, 10, 100)}
t_start = time.time()
grid_search = dcv.GridSearchCV(lr, parameters, scoring="roc_auc", cv=skf)
grid_search.fit(Xvect, y_text)
t_end = time.time()
print(
    f"Elapsed time for grid_search (without time spended to vectorization) {round((t_end - t_start))} (s):"
)

In [ ]:
grid_search.best_score_


حاولت معرفة مدى جودة dask مع الغابة العشوائية ذات المعلمات الأصلية، ولكن في بعض الأحيان يظهر هذا الخطأ "(OSError: [Errno 24] عدد كبير جدًا من الملفات المفتوحة) بعد التنفيذ، ولم أتمكن من إصلاحه...." في بعض الأحيان يعمل بشكل جيد، وبالنسبة للبيانات الصغيرة فإنه يعمل في معظم الحالات، ولكن إذا قمت بإعادة تشغيل دفتر الملاحظات هذا عدة مرات، فهناك فرصة كبيرة للحصول على مثل هذا الخطأ. لذا، أعتقد أن dask-ml مفيد جدًا، ولكنني بالتأكيد لا أعرف كيف يجب استخدامه بشكل صحيح. 


In [ ]:
from sklearn.ensemble import RandomForestClassifier

rf = RandomForestClassifier(random_state=17)
min_samples_leaf = [1, 2, 3]
max_features = [0.3, 0.5, 0.7]
max_depth = [None]

parameters = {
    "max_features": max_features,
    "min_samples_leaf": min_samples_leaf,
    "max_depth": max_depth,
}
grid_search = dcv.GridSearchCV(rf, parameters, scoring="roc_auc", cv=skf)
t_start = time.time()
grid_search.fit(Xvect, y_text)
t_end = time.time()
print(
    f"Elapsed time for dask grid_search for Random Forest {round((t_end - t_start))} (s):"
)


### التعامل مع النموذج الذي يحتوي على بيانات كبيرة
هناك عدد من النماذج المعاد كتابتها في dask، والتي يمكن أن تأخذ كائن dask (مصفوفات ضخمة) وتحسب النماذج عليها. يمكنك قراءة المزيد في وثائق dask . فيما يلي مثال على KMeans، ولكن هناك أيضًا نسخة dask من النماذج الخطية ووظائف المعالجة. الترميز مشابه جدًا لـ scikit-Learn، ويجب أن يكون سهل الاستخدام. 


In [ ]:
from dask_ml import datasets
from dask_ml.cluster import KMeans

In [ ]:
X, y = datasets.make_blobs(
    n_samples=10000000, chunks=1000000, random_state=0, centers=3
)
# Persist will give you back a lazy dask.delayed object
X = X.persist()
X

In [ ]:
km = KMeans(n_clusters=3, init_max_iter=2, oversampling_factor=10)
km.fit(X)

في الواقع، لقد قرأت المقال عن dask منذ يومين وقررت أن هذه المهمة من خلال البرنامج التعليمي هي طريقة جيدة للتعرف على المكتبة. لذا أطلب منك ألا تكون صارمًا جدًا إذا أساءت فهم شيء ما :))